---
title: "M6 — Semantic Evaluation Suite: STS + Clustering + Cross-model Comparison"
date: "2026-03-17"
author: "Person B"
project: "Sentimind — Mental Health Sentiment Analysis"
purpose: >
  Đánh giá chiều sâu ngữ nghĩa của tập dữ liệu và kết quả phân loại thông qua
  Semantic Textual Similarity (STS within/cross-class), phân cụm embedding
  (UMAP + HDBSCAN), và bảng so sánh tổng hợp 3 model (BiLSTM, BERTweet, Gemini).
objectives:
  - Sinh sentence embeddings từ test.csv bằng all-MiniLM-L6-v2 (cache .npy)
  - Tính STS within-class và cross-class cho tất cả 7 nhãn
  - Giảm chiều UMAP 2D và phân cụm HDBSCAN
  - Xuất biểu đồ semantic_cluster_plot.png (ground-truth vs discovered clusters)
  - Tải bilstm_metrics.json + bertweet_metrics.json + llm_metrics.json
  - Tạo comparison_report.json sắp xếp theo macro_f1
  - Phân tích sâu các class bị nhầm lẫn nhiều nhất
milestone: M6
depends_on: "data/processed/test.csv (M2)"
optional_depends_on: "bilstm_metrics.json (M3), bertweet_metrics.json (M4), llm_metrics.json (M5)"
outputs:
  - data/artifacts/sts_report.json
  - data/artifacts/semantic_cluster_plot.png
  - data/artifacts/semantic_embeddings_2d.npy
  - data/artifacts/comparison_report.json
---

# M6 — Semantic Evaluation Suite

> **Mục đích:** Đánh giá ngữ nghĩa: STS, phân cụm UMAP/HDBSCAN, và so sánh tổng hợp 3 model.
> Có thể chạy `--skip-comparison` nếu chưa có `llm_metrics.json`.

**Cài đặt trước:**
```bash
pip install sentence-transformers umap-learn hdbscan
```

In [1]:
%pip install sentence-transformers umap-learn hdbscan

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import json
import subprocess
import shlex
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"
print(f"Project root  : {PROJECT_ROOT}")
print(f"Artifacts dir : {ARTIFACTS_DIR}")

Project root  : D:\sentimind
Artifacts dir : D:\sentimind\data\artifacts


## 1. Chạy STS + Clustering (không cần LLM metrics)

In [3]:
import importlib
import json
from pathlib import Path

import pandas as pd
import yaml

import scripts.run_semantic_analysis as semantic_mod
importlib.reload(semantic_mod)
from scripts.run_semantic_analysis import (
    generate_embeddings,
    run_clustering,
    run_comparison,
    run_sts,
)

cfg_path = PROJECT_ROOT / "configs" / "semantic.yaml"
with open(cfg_path, "r", encoding="utf-8") as f:
    semantic_cfg = yaml.safe_load(f)

artifacts_dir = PROJECT_ROOT / semantic_cfg["output"]["artifacts_dir"]
artifacts_dir.mkdir(parents=True, exist_ok=True)

data_cfg = semantic_cfg["data"]
df = pd.read_csv(PROJECT_ROOT / data_cfg["test_path"])
sample_size = data_cfg.get("sample_size")
if sample_size and sample_size < len(df):
    df = df.sample(n=sample_size, random_state=data_cfg.get("sample_seed", 42)).reset_index(drop=True)

texts = df[data_cfg["text_col"]].astype(str).tolist()
labels = df[data_cfg["label_col"]].astype(int).tolist()

with open(PROJECT_ROOT / "configs" / "preprocessing.yaml", "r", encoding="utf-8") as f:
    pre_cfg = yaml.safe_load(f)

raw_map = pre_cfg.get("label_map", {})
label_map = {}
for name, label_id in raw_map.items():
    if label_id not in label_map:
        label_map[label_id] = name.title()

emb_cfg = semantic_cfg["embeddings"]
cache_path = PROJECT_ROOT / emb_cfg.get("cache_path", "data/artifacts/semantic_embeddings.npy")
embeddings = generate_embeddings(
    texts=texts,
    model_name=emb_cfg["model_name"],
    batch_size=emb_cfg.get("batch_size", 64),
    device=emb_cfg.get("device"),
    cache_path=cache_path,
)
print(f"Embeddings shape: {embeddings.shape}")

sts_cfg = semantic_cfg.get("sts", {})
sts_report = run_sts(
    embeddings=embeddings,
    labels=labels,
    label_map=label_map,
    pairs_per_class=sts_cfg.get("pairs_per_class", 50),
    seed=sts_cfg.get("seed", semantic_cfg.get("seed", 42)),
)
sts_path = artifacts_dir / semantic_cfg["output"]["sts_report_name"]
with open(sts_path, "w", encoding="utf-8") as f:
    json.dump(sts_report, f, indent=2, ensure_ascii=False)

cluster_summary = run_clustering(
    embeddings=embeddings,
    labels=labels,
    label_map=label_map,
    umap_cfg=semantic_cfg["clustering"]["umap"],
    hdbscan_cfg=semantic_cfg["clustering"]["hdbscan"],
    artifacts_dir=artifacts_dir,
    plot_name=semantic_cfg["output"]["cluster_plot_name"],
    embeddings_2d_name=semantic_cfg["output"]["embeddings_2d_name"],
    seed=semantic_cfg.get("seed", 42),
)

comparison_cfg = {
    name: str(PROJECT_ROOT / path)
    for name, path in semantic_cfg.get("comparison", {}).items()
    if path and (PROJECT_ROOT / path).exists()
}
comparison_report = run_comparison(
    comparison_cfg=comparison_cfg,
    artifacts_dir=artifacts_dir,
    out_name=semantic_cfg["output"]["comparison_report_name"],
)

print("\nSemantic analysis complete.")
print(f"- STS report         : {sts_path}")
print(f"- Cluster plot       : {artifacts_dir / semantic_cfg['output']['cluster_plot_name']}")
print(f"- 2D embeddings      : {artifacts_dir / semantic_cfg['output']['embeddings_2d_name']}")
print(f"- Comparison report  : {artifacts_dir / semantic_cfg['output']['comparison_report_name']}")
print(f"- Models compared    : {', '.join(comparison_report.get('models', {}).keys()) or 'none'}")
print(f"- Ranking by Macro F1: {comparison_report.get('ranking_by_macro_f1', [])}")
print(f"- Clusters found     : {cluster_summary.get('n_clusters', 'N/A')}")
print(f"- Noise ratio        : {cluster_summary.get('noise_ratio', 'N/A')}")
print(f"- Cross-class STS    : {sts_report.get('cross_class_avg_cosine', 'N/A')}")

21:29:05 | INFO     | scripts.run_semantic_analysis | Loading cached embeddings from D:\sentimind\data\artifacts\semantic_embeddings.npy.


Embeddings shape: (10701, 384)


c:\Users\Celine\miniconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
21:29:19 | INFO     | scripts.run_semantic_analysis | Running UMAP dimensionality reduction...
c:\Users\Celine\miniconda3\envs\nlp\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
21:30:06 | INFO     | scripts.run_semantic_analysis | 2-D embeddings saved to D:\sentimind\data\artifacts\semantic_embeddings_2d.npy.
21:30:06 | INFO     | scripts.run_semantic_analysis | Running HDBSCAN clustering...
21:30:06 | INFO     | scripts.run_semantic_analysis | HDBSCAN: 6 clusters found, noise_ratio=0.197
21:30:07 | INFO     | scripts.run_semantic_analysis | Cluster plot saved to D:\sentimind\data\artifacts\semantic_cluster_plot.png.
21:30:07 | INFO   


Semantic analysis complete.
- STS report         : D:\sentimind\data\artifacts\sts_report.json
- Cluster plot       : D:\sentimind\data\artifacts\semantic_cluster_plot.png
- 2D embeddings      : D:\sentimind\data\artifacts\semantic_embeddings_2d.npy
- Comparison report  : D:\sentimind\data\artifacts\comparison_report.json
- Models compared    : bilstm_metrics, bertweet_metrics
- Ranking by Macro F1: ['bilstm_metrics', 'bertweet_metrics']
- Clusters found     : 6
- Noise ratio        : 0.1968
- Cross-class STS    : 0.1405


## 2. Xem STS Report

In [6]:
import pandas as pd
from IPython.display import display, Image

sts_path = ARTIFACTS_DIR / "sts_report.json"
if sts_path.exists():
    with open(sts_path, "r", encoding="utf-8") as f:
        sts = json.load(f)

    within_scores = sts.get("within_class_avg_cosine", {})
    print("STS Within-class (cosine similarity trung bình trong cùng nhãn):")
    rows = [{"Class": cls, "Within-class STS": score} for cls, score in within_scores.items()]
    if rows:
        display(
            pd.DataFrame(rows)
            .set_index("Class")
            .sort_values("Within-class STS", ascending=False)
            .round(4)
        )

    cross_class = sts.get("cross_class_avg_cosine")
    print("\nSTS Cross-class mean (trung bình giữa các cặp nhãn khác nhau):")
    print(f"  {cross_class:.4f}" if cross_class is not None else "  N/A")

    if within_scores and cross_class is not None:
        best_class, best_score = max(within_scores.items(), key=lambda item: item[1])
        print(f"\nClass cohesion cao nhất: {best_class} ({best_score:.4f})")
        print(f"Within/Cross gap      : {best_score - cross_class:.4f}")
else:
    print("sts_report.json chưa có. Chạy Cell 5 trước.")

STS Within-class (cosine similarity trung bình trong cùng nhãn):


,Within-class STS
Class,
Suicidal,0.2958
Depression,0.2836
Bipolar,0.2380
Anxiety,0.2209
Personality Disorder,0.1818
Stress,0.1553
Normal,0.0832



STS Cross-class mean (trung bình giữa các cặp nhãn khác nhau):
  0.1405

Class cohesion cao nhất: Suicidal (0.2958)
Within/Cross gap      : 0.1553


## 3. Xem comparison_report.json (sau khi có đủ metrics M3/M4/M5)

In [7]:
cmp_path = ARTIFACTS_DIR / "comparison_report.json"
model_name_map = {
    "bilstm_metrics": "BiLSTM",
    "bertweet_metrics": "BERTweet",
    "llm_metrics": "LLM",
}

if cmp_path.exists():
    with open(cmp_path, "r", encoding="utf-8") as f:
        cmp = json.load(f)

    model_rows = []
    for model_key, metrics in cmp.get("models", {}).items():
        model_rows.append({
            "model": model_name_map.get(model_key, model_key),
            "accuracy": metrics.get("accuracy"),
            "macro_f1": metrics.get("macro_f1"),
            "weighted_f1": metrics.get("weighted_f1"),
        })

    if model_rows:
        df_cmp = pd.DataFrame(model_rows).sort_values("macro_f1", ascending=False)
        print("So sánh tổng hợp (sắp xếp theo Macro F1):")
        display(df_cmp.round(4))

        ranking = [model_name_map.get(name, name) for name in cmp.get("ranking_by_macro_f1", [])]
        print(f"Ranking: {ranking}")
    else:
        print("comparison_report.json có tồn tại nhưng không có dữ liệu model.")
else:
    print("comparison_report.json chưa có.")
    print("   Chạy lại Cell 5 để dựng comparison trực tiếp từ bilstm_metrics.json, bertweet_metrics.json, llm_metrics.json.")

So sánh tổng hợp (sắp xếp theo Macro F1):


,model,accuracy,macro_f1,weighted_f1
0,BiLSTM,0.8244,0.8359,0.8248
1,BERTweet,0.8210,0.8268,0.8209


Ranking: ['BiLSTM', 'BERTweet']
